# Author Metadata and Wikidata Matching

This notebook classifies author-nationality provenance and matches Project Gutenberg author records to Wikidata using progressively broader normalized-name keys. It retains both the initial and revised matching approaches, together with diagnostics for ambiguous and unmatched authors. Historical local inputs and review-oriented outputs are preserved as research provenance.


In [24]:
import pandas as pd
import re

In [25]:
all_writers = pd.read_csv('all_writers.csv')
all_writers

,writer,place,gender,linkcount
0,William Shakespeare,Kingdom of England,male,335
1,Donald Trump,United States,male,331
2,Albert Einstein,Cisleithania,male,320
3,Albert Einstein,Kingdom of Württemberg,male,320
4,Albert Einstein,statelessness,male,320
...,...,...,...,...
35318,Dean Spade,United States,trans man,11
35319,Karl M. Baer,Germany,intersex trans man,11
35320,Karl M. Baer,Israel,intersex trans man,11
35321,Karl M. Baer,Principality of Waldeck,intersex trans man,11


In [27]:
places = pd.Series(all_writers["place"].dropna().unique()).sort_values()
print(f"共有 {len(places)} 个不同的国籍取值")

共有 820 个不同的国籍取值


## Nationality classification


In [28]:
# Stage 2: keyword matching
INCLUDE = re.compile(r"""
    england | scotland | wales | britain | british
  | ireland | irish
  | united \s+ kingdom | \b uk \b
  | isle \s+ of \s+ man | channel \s+ islands | guernsey | \b jersey \b
""", re.I | re.X)

EXCLUDE = re.compile(r"""
    new \s+ england
  | new \s+ south \s+ wales
  | new \s+ jersey
  | united \s+ kingdom \s+ of \s+ the \s+ netherlands
  | british \s+ america                      # 北美殖民地
  | british \s+ national                     # BN(O)，香港
  | british \s+ rule \s+ in                  # British rule in Myanmar 等
  | british \s+ (?: raj | india | ceylon | malaya | burma | hong \s+ kong
                 | guiana | honduras | somaliland | mandate | palestine
                 | east \s+ africa | west \s+ africa | nigeria
                 | new \s+ guinea | virgin \s+ islands | columbia
                 | empire | overseas )
  | colony \s+ of
""", re.I | re.X)

hit     = places[places.str.contains(INCLUDE) & ~places.str.contains(EXCLUDE)]
dropped = places[places.str.contains(INCLUDE) &  places.str.contains(EXCLUDE)]

print("\n=== 命中（请核对）===")
for p in hit:
    print(" ", p)
print("\n=== 被黑名单排除（请核对是否排错）===")
for p in dropped:
    print(" ", p)


=== 命中（请核对）===
  Commonwealth of England
  England
  Gaelic Ireland
  Ireland
  Irish Free State
  Irish Republic
  Isle of Man
  Kingdom of England
  Kingdom of Great Britain
  Kingdom of Ireland
  Kingdom of Scotland
  Scotland
  United Kingdom
  United Kingdom of Great Britain and Ireland
  Wales
  sub-Roman Britain

=== 被黑名单排除（请核对是否排错）===
  British America
  British Ceylon
  British Empire
  British Hong Kong
  British Malaya
  British National (Overseas)
  British Raj
  British rule in Myanmar
  Colony of New South Wales


In [29]:
import re
import pandas as pd

# Rule order matters; the first matching rule is applied.
TIER_RULES = [
    # 1. Exclude unrelated values that merely contain a keyword.
    ("not_uk", r"""
        united \s+ kingdom \s+ of \s+ the \s+ netherlands
      | \b new \s+ jersey \b
    """),
    # 2. Exclude settler colonies and colonial North America.
    ("settler", r"""
        british \s+ america | british \s+ columbia
      | new \s+ england | new \s+ south \s+ wales
      | \b australia | \b canada | new \s+ zealand
      | cape \s+ colony | \b natal \b | \b transvaal \b
    """),
    # 3. Include administrative-colony cases for review.
    ("colonial", r"""
        british \s+ (?: raj | india | ceylon | malaya | burma
                      | hong \s+ kong | somaliland | guiana | honduras
                      | empire | rule | national | mandate | overseas )
      | colonial \s+ (?: nigeria | kenya | rhodesia )
      | straits \s+ settlements
    """),
    # 4. Include home nations and Crown dependencies; Northern Ireland belongs here.
    ("core", r"""
        northern \s+ ireland
      | \b england \b | scotland | \b wales \b | britain | british
      | united \s+ kingdom | \b uk \b
      | isle \s+ of \s+ man | channel \s+ islands | guernsey | \b jersey \b
    """),
    # 5. Keep Ireland as a separate tier.
    ("ireland", r" ireland | irish "),
]
TIER_RULES = [(t, re.compile(p, re.I | re.X)) for t, p in TIER_RULES]

def to_tier(place):
    if pd.isna(place):
        return None
    for tier, pat in TIER_RULES:
        if pat.search(str(place)):
            return None if tier == "not_uk" else tier
    return None

all_writers["tier"] = all_writers["place"].apply(to_tier)

In [30]:
print(all_writers[all_writers.tier.notna()]
      .groupby(["tier", "place"]).size())

tier      place                                      
colonial  British Ceylon                                    2
          British Empire                                    7
          British Hong Kong                                 7
          British Malaya                                    1
          British National (Overseas)                       1
          British Raj                                     451
          British rule in Myanmar                           1
          Colonial Nigeria                                  3
core      Commonwealth of England                           1
          England                                          21
          Isle of Man                                       1
          Kingdom of England                              197
          Kingdom of Great Britain                        280
          Kingdom of Scotland                              18
          Scotland                                         17
          United

In [31]:
# Treat core and Ireland cases as confirmed; retain colonial and settler cases for separate review.
confirmed = tiers[tiers.apply(lambda t: bool(t & {"core", "ireland"}))]
candidates = tiers[tiers.apply(
    lambda t: bool(t & {"colonial", "settler"}) and not (t & {"core", "ireland"})
)]
print(f"确定收录 {len(confirmed)}，待裁定 {len(candidates)}")

确定收录 3243，待裁定 1186


In [41]:
uk = all_writers.dropna(subset=["tier"]).copy()
uk = uk[uk.tier != "settler"]              # Exclude settler-colony cases.

per_writer = (uk.groupby("writer")
                .agg(tiers=("tier", lambda s: sorted(set(s))),
                     places=("place", lambda s: sorted(set(s))),
                     gender=("gender", "first"))
                .reset_index())

# Derived flag for concise filtering.
per_writer["colonial_only"] = per_writer.tiers.apply(lambda t: t == ["colonial"])
per_writer["nationality"] = per_writer.tiers.apply(
    lambda t: "Irish" if t == ["ireland"] else "British")

#per_writer.to_csv("uk_writers_with_provenance.csv", index=False)
#print(per_writer.colonial_only.value_counts())

In [42]:
per_writer

,writer,tiers,places,gender,colonial_only,nationality
0,A. A. Milne,[core],[United Kingdom],male,False,British
1,A. C. Benson,[core],[United Kingdom of Great Britain and Ireland],male,False,British
2,A. C. Bhaktivedanta Swami Prabhupada,[colonial],[British Raj],male,True,British
3,A. C. Bradley,[core],"[United Kingdom, United Kingdom of Great Brita...",male,False,British
4,A. J. Ayer,[core],[United Kingdom],male,False,British
...,...,...,...,...,...,...
3699,Zoltan Korda,[core],"[United Kingdom, United Kingdom of Great Brita...",male,False,British
3700,Zoë Heller,[core],[United Kingdom],female,False,British
3701,Zygmunt Bauman,[core],[United Kingdom],male,False,British
3702,Ælfric of Eynsham,[core],[Kingdom of England],male,False,British


## Author-name matching


In [38]:
gb = pd.read_csv('pg_candidate_v3.csv')

In [39]:
print(gb[["author_raw", "author_display",
          "author_key_normalized", "author_display_normalized"]].head(10).to_string())

                                                          author_raw            author_display     author_key_normalized author_display_normalized
0                                        Melville, Herman, 1819-1891           Herman Melville           herman melville           herman melville
1                                           Cather, Willa, 1873-1947              Willa Cather              willa cather              willa cather
2                                           Hardy, Thomas, 1840-1928              Thomas Hardy              thomas hardy              thomas hardy
3                               Gilman, Charlotte Perkins, 1860-1935  Charlotte Perkins Gilman  charlotte perkins gilman  charlotte perkins gilman
4                                    Hawthorne, Nathaniel, 1804-1864       Nathaniel Hawthorne       nathaniel hawthorne       nathaniel hawthorne
5                           Wells, H. G. (Herbert George), 1866-1946               H. G. Wells      herbert george wel

In [43]:
import re, unicodedata
import pandas as pd

SUFFIX = re.compile(r"\b(?:jr|sr|phd|md|sir|dame|lord|lady|rev|dr|mrs|mr|ms|"
                    r"i{1,3}|iv|v|vi{1,3})\b")

def norm(name):
    """小写、去重音、去括号内容、去生卒年、去标点、压空格"""
    if pd.isna(name):
        return ""
    s = str(name)
    s = unicodedata.normalize("NFKD", s)
    s = "".join(c for c in s if not unicodedata.combining(c))
    s = s.lower()
    s = re.sub(r"\([^)]*\)", " ", s)          # Remove parenthetical annotations.
    s = re.sub(r"\b\d{3,4}\s*[-?]\s*\d{0,4}\b", " ", s)  # Remove lifespan ranges such as 1819–1891.
    s = re.sub(r"[^a-z\s,]", " ", s)          # Preserve commas for later surname-first detection.
    s = SUFFIX.sub(" ", s)
    return re.sub(r"\s+", " ", s).strip(" ,")

def key_sorted(name):
    """把 token 排序，让 'Melville, Herman' 和 'Herman Melville' 落到同一个 key"""
    n = norm(name).replace(",", " ")
    toks = [t for t in n.split() if len(t) > 1]   # Drop single-letter initials such as H.
    return " ".join(sorted(toks))

# Build matching keys for both datasets.
gb = gutenberg.copy()
gb["k_display"] = gb["author_display"].apply(key_sorted)
gb["k_raw"]     = gb["author_raw"].apply(key_sorted)

wd = per_writer.copy()
wd["k"] = wd["writer"].apply(key_sorted)
wd = wd[wd.k != ""]

# Inspect Wikidata keys that map to multiple people.
dup = wd[wd.k.duplicated(keep=False)].sort_values("k")
print(f"Wikidata 侧重复 key：{len(dup)} 条")
print(dup[["writer", "k", "tiers"]].head(20).to_string())

wd_lookup = wd.drop_duplicates("k").set_index("k")

# Match display names first, then fall back to raw names.
gb["matched_key"] = gb["k_display"].where(gb["k_display"].isin(wd_lookup.index))
gb["matched_key"] = gb["matched_key"].fillna(
    gb["k_raw"].where(gb["k_raw"].isin(wd_lookup.index)))

gb["wd_writer"]     = gb.matched_key.map(wd_lookup["writer"])
gb["wd_tiers"]      = gb.matched_key.map(wd_lookup["tiers"])
gb["wd_nationality"]= gb.matched_key.map(wd_lookup["nationality"])
gb["colonial_only"] = gb.matched_key.map(wd_lookup["colonial_only"])

Wikidata 侧重复 key：7 条
              writer         k   tiers
886      E. L. James     james  [core]
2211     M. R. James     james  [core]
2610     P. D. James     james  [core]
718   D. H. Lawrence  lawrence  [core]
3311  T. E. Lawrence  lawrence  [core]
719     D. M. Thomas    thomas  [core]
2794    R. S. Thomas    thomas  [core]


In [45]:
print(len(per_writer), per_writer.writer.isna().sum(),
      (per_writer.writer.astype(str).str.strip() == "").sum())
print(per_writer.writer.sample(15).tolist())
print(gb.author_display_normalized.isna().sum(), gb.author_display.nunique())

3704 0 0
['E. M. Forster', 'Graham Joyce', 'Malachy McCourt', 'Bridget Dowling', 'Jonathan Coe', 'G. E. M. Anscombe', 'Nigel Planer', 'Shashi Deshpande', 'Sukumar Sen', 'H. G. Wells', 'Danny Boyle', 'Ian Watson', 'Lee Hall', 'Victor Gollancz', 'Henry Wotton']
0 1906


In [46]:
import re, unicodedata
import pandas as pd

HONOR = re.compile(r"\b(?:sir|dame|lord|lady|rev|dr|mrs|mr|ms|jr|sr|"
                   r"baron|baroness|earl|viscount|st)\b")

def norm(s):
    if pd.isna(s):
        return ""
    s = unicodedata.normalize("NFKD", str(s))
    s = "".join(c for c in s if not unicodedata.combining(c))
    s = s.lower()
    s = re.sub(r"\[[^\]]*\]", " ", s)                    # [Illustrator]
    s = re.sub(r"\d{3,4}\s*[-?]\s*\d{0,4}", " ", s)      # 1819-1891
    s = re.sub(r"[^a-z\s]", " ", s)
    s = HONOR.sub(" ", s)
    return re.sub(r"\s+", " ", s).strip()

def flip_raw(raw):
    """'Dickens, Charles, 1812-1870; Leech...' -> 'charles dickens'"""
    if pd.isna(raw):
        return ""
    first = str(raw).split(";")[0]
    parts = [p.strip() for p in first.split(",")]
    if len(parts) >= 2:
        return norm(f"{parts[1]} {parts[0]}")
    return norm(first)

# Three matching-key levels
def k_full(n):        # Sort all tokens, including initials.
    t = [x for x in norm(n).split() if x]
    return " ".join(sorted(t)) if len(t) >= 2 else ""

def k_surinit(n):     # Surname plus given-name initial handles H. G. / Herbert George.
    t = norm(n).split()
    return f"{t[-1]}|{t[0][0]}" if len(t) >= 2 else ""

def k_sur(n):         # Surname-only matching is reported separately and not accepted automatically.
    t = norm(n).split()
    return t[-1] if len(t) >= 2 else ""

KEYFUNCS = [("full", k_full), ("surinit", k_surinit), ("sur", k_sur)]

In [47]:
gb = gutenberg.copy()
gb["cand"] = [
    {v for v in (norm(a), norm(b), flip_raw(c)) if v}
    for a, b, c in zip(gb.author_key_normalized,
                       gb.author_display_normalized,
                       gb.author_raw)
]

wd = per_writer.dropna(subset=["writer"]).copy()
wd = wd[wd.writer.astype(str).str.strip() != ""]

# Build one lookup per level and mark keys shared by multiple people as ambiguous.
lookups = {}
for lvl, f in KEYFUNCS:
    wd[f"k_{lvl}"] = wd.writer.apply(f)
    sub = wd[wd[f"k_{lvl}"] != ""]
    grp = sub.groupby(f"k_{lvl}")["writer"].apply(list)
    lookups[lvl] = grp

def match(cands):
    for lvl, f in KEYFUNCS:
        for c in cands:
            k = f(c)
            if k and k in lookups[lvl].index:
                hits = lookups[lvl][k]
                return pd.Series([hits[0], lvl, len(hits)])
    return pd.Series([None, None, 0])

gb[["wd_writer", "match_level", "n_cand"]] = gb.cand.apply(match)

info = wd.drop_duplicates("writer").set_index("writer")
for col in ["tiers", "places", "nationality", "colonial_only"]:
    gb[f"wd_{col}"] = gb.wd_writer.map(info[col])

In [48]:
au = gb.drop_duplicates("author_display")
print(f"作者 {au.wd_writer.notna().sum()}/{len(au)} 匹配上")
print(f"书   {gb.wd_writer.notna().sum()}/{len(gb)}")
print("\n分层命中（作者数）:")
print(au.match_level.value_counts(dropna=False))

# Require manual review for surname-only or ambiguous-key matches.
print("\n需复核（sur 层 或 多候选）:")
print(au[(au.match_level == "sur") | (au.n_cand > 1)]
      [["author_display", "wd_writer", "match_level", "n_cand"]]
      .head(50).to_string())

# Sort unmatched authors by book count to prioritize likely missed British writers.
print("\n未匹配 top40:")
print(gb[gb.wd_writer.isna()].groupby("author_display")
        .size().sort_values(ascending=False).head(40).to_string())

作者 855/1906 匹配上
书   3647/6618

分层命中（作者数）:
match_level
NaN        1051
sur         534
full        183
surinit     138
Name: count, dtype: int64

需复核（sur 层 或 多候选）:
                   author_display                                  wd_writer match_level  n_cand
0                 Herman Melville                           Pauline Melville         sur     1.0
15                  Stephen Crane                               Walter Crane         sur     1.0
42           Gene Stratton-Porter                                Jane Porter         sur     3.0
60                  Edith Wharton        Philip Wharton, 1st Duke of Wharton         sur     1.0
63             Christopher Morley                                John Morley         sur     3.0
78            William Wells Brown                              Christy Brown         sur     9.0
91                   John Fox Jr.                                 George Fox         sur     2.0
99              Eleanor H. Porter                            

## Revised nationality classification


In [99]:
df1 = pd.read_csv('query (6).csv')
df2 = pd.read_csv('query (4).csv')
df3 = pd.read_csv('query (5).csv')
result = pd.concat([df1, df2], ignore_index=True)

In [102]:
novelists = pd.concat([df1, df2, df3], ignore_index=True)
novelists

,s,writer,birthYear,deathYear,place,gender,linkcount
0,http://www.wikidata.org/entity/Q77413,Charlotte von Kalb,1761,1843.0,Kingdom of Bavaria,female,16
1,http://www.wikidata.org/entity/Q77411,Gisela von Arnim,1827,1889.0,Kingdom of Prussia,female,19
2,http://www.wikidata.org/entity/Q77926,Dorothea Viehmann,1755,1816.0,Electorate of Hesse,female,19
3,http://www.wikidata.org/entity/Q77483,Marie Hankel,1844,1929.0,Germany,female,20
4,http://www.wikidata.org/entity/Q77271,Dorothea von Schlegel,1764,1839.0,Germany,female,21
...,...,...,...,...,...,...,...
79138,http://www.wikidata.org/entity/Q104814977,Max Gissen,1909,1985.0,NaN,male,1
79139,http://www.wikidata.org/entity/Q106410197,Nanina Alba,1915,1968.0,NaN,female,1
79140,http://www.wikidata.org/entity/Q106447866,Myra Cohn Livingston,1926,1996.0,United States,female,1
79141,http://www.wikidata.org/entity/Q7285337,Raj & DK,1901,NaN,NaN,NaN,8


In [103]:
places = pd.Series(novelists["place"].dropna().unique()).sort_values()
print(f"共有 {len(places)} 个不同的国籍取值")

共有 643 个不同的国籍取值


In [104]:
# ── Stage 1: overrides ──────────────────────────────────────────
# These contain UK keywords as substrings but aren't UK. They must be
# evaluated before the core/settler rules or they'd be misclassified.
OVERRIDES = [
    ("europe",  r" united \s+ kingdom \s+ of \s+ the \s+ netherlands "),
    ("america", r" \b new \s+ jersey \b | new \s+ england | \b new \s+ york \b "),
    ("settler", r" new \s+ south \s+ wales | british \s+ columbia "),
]

# ── Stage 2: main tiers (order matters) ─────────────────────────
TIER_RULES = OVERRIDES + [
    # settler colonies + colonial North America → excluded from UK, kept for audit
    ("settler", r"""
        british \s+ america
      | \b australia | \b canada | new \s+ zealand
      | cape \s+ colony | \b natal \b | \b transvaal \b
      | dominion \s+ of | newfoundland | \b quebec \b | \b ontario \b
      | south \s+ africa | orange \s+ free \s+ state
    """),
    # administrative colonies → included as British, flagged for review
    ("colonial", r"""
        british \s+ (?: raj | india | ceylon | malaya | burma
                      | hong \s+ kong | somaliland | guiana | honduras
                      | empire | rule | national | mandate | overseas )
      | colonial \s+ (?: nigeria | kenya | rhodesia )
      | straits \s+ settlements
    """),
    # home nations + crown dependencies (Northern Ireland belongs here)
    ("core", r"""
        northern \s+ ireland
      | \b england \b | scotland | \b wales \b | britain | british
      | united \s+ kingdom | \b uk \b
      | isle \s+ of \s+ man | channel \s+ islands | guernsey | \b jersey \b
    """),
    ("ireland", r" ireland | irish "),

    # ── non-UK, retained and labelled ──
    ("america", r"""
        united \s+ states | \b usa \b | \b u\.s\. | confederate \s+ states
      | thirteen \s+ colonies
    """),
    ("europe", r"""
        \b france | \b french | german | prussia | austria | \b russia | soviet
      | \b italy | italian | \b spain | spanish | netherlands | \b dutch
      | sweden | norway | denmark | finland | iceland | switzerland | swiss
      | poland | polish | belgium | hungary | bohemia | czech | slovak
      | \b greece | portugal | romania | serbia | croatia | ukraine | bulgaria
      | estonia | latvia | lithuania | luxembourg | \b malta
      | ottoman | yugoslavia | savoy | \b naples | papal \s+ states | tuscany
    """),
]
TIER_RULES = [(t, re.compile(p, re.I | re.X)) for t, p in TIER_RULES]

def to_tier(place):
    if pd.isna(place) or str(place).strip() == "":
        return "no_p27"
    s = str(place)
    for tier, pat in TIER_RULES:
        if pat.search(s):
            return tier
    return "other"          # has a P27 value, just not one we've bucketed

novelists["tier"] = novelists["place"].apply(to_tier)

UK_TIERS = {"core", "ireland", "colonial"}
novelists["is_uk"] = novelists.tier.isin(UK_TIERS)

In [105]:
novelists['tier'].value_counts()

,count
tier,
europe,29078
america,12983
other,12117
core,10975
no_p27,7743
settler,3664
colonial,1991
ireland,592


In [106]:
TIER_PRIORITY = ["core", "ireland", "colonial", "settler",
                 "america", "europe", "other", "no_p27"]
RANK = {t: i for i, t in enumerate(TIER_PRIORITY)}

per_writer = (
    novelists
      .assign(_r=novelists.tier.map(RANK))
      .sort_values("_r")
      .groupby("s", as_index=False)
      .agg(writer=("writer", "first"),
           birthYear=("birthYear", "first"),
           deathYear=("deathYear", "first"),
           gender=("gender", "first"),
           linkcount=("linkcount", "max"),
           tier=("tier", "first"),                       # highest-priority tier
           all_tiers=("tier", lambda s: sorted(set(s))),
           places=("place", lambda s: sorted({str(p) for p in s if pd.notna(p)})))
)

per_writer["is_uk"] = per_writer.tier.isin(UK_TIERS)
per_writer["colonial_only"] = per_writer.all_tiers.apply(lambda t: t == ["colonial"])
per_writer["nationality"] = per_writer.tier.map(
    lambda t: "Irish" if t == "ireland" else ("British" if t in UK_TIERS else None))

print(per_writer.tier.value_counts())
print(f"\ntotal {len(per_writer)}, UK {per_writer.is_uk.sum()}")
per_writer.to_csv("wd_novelists_tiered.csv", index=False)

tier
europe      22474
america     12397
core         7947
no_p27       7480
other        6535
settler      2300
colonial     1909
ireland       403
Name: count, dtype: int64

total 61445, UK 10259


In [91]:
gutenberg.head(20)

,gutenberg_id,title,subjects,subject_text,author_raw,author_display,author_query,is_domestic,is_detective_mystery,is_science_fiction,...,genre_tags,broad_group,author_key_normalized,author_display_normalized,is_canonical_author,canonical_period,is_english_fiction_19c,is_general_19c_candidate,auto_group,has_fiction_signal
0,15,"Moby-Dick; or, The Whale",Whaling -- Fiction; Sea stories; Psychological...,whaling -- fiction; sea stories; psychological...,"Melville, Herman, 1819-1891",Herman Melville,Herman Melville,0,0,0,...,adventure,major_genre_non_domestic,herman melville,herman melville,0,NaN,0,0,non_domestic_genre,True
1,24,O Pioneers!,Nebraska -- Fiction; Historical fiction; Front...,nebraska -- fiction; historical fiction; front...,"Cather, Willa, 1873-1947",Willa Cather,Willa Cather,1,0,0,...,historical,domestic_and_genre,willa cather,willa cather,0,NaN,0,0,domestic_adjacent_hybrid,True
2,27,Far from the Madding Crowd,Didactic fiction; Love stories; Triangles (Int...,didactic fiction; love stories; triangles (int...,"Hardy, Thomas, 1840-1928",Thomas Hardy,Thomas Hardy,0,0,0,...,NaN,other_non_domestic_non_major_genre,thomas hardy,thomas hardy,1,Victorian,0,0,canonical_other,True
3,32,Herland,Utopias -- Fiction; Women -- Fiction; Utopian ...,utopias -- fiction; women -- fiction; utopian ...,"Gilman, Charlotte Perkins, 1860-1935",Charlotte Perkins Gilman,Charlotte Perkins Gilman,0,0,1,...,science_fiction,major_genre_non_domestic,charlotte perkins gilman,charlotte perkins gilman,0,NaN,0,0,non_domestic_genre,True
4,33,The Scarlet Letter,Adultery -- Fiction; Historical fiction; Reven...,adultery -- fiction; historical fiction; reven...,"Hawthorne, Nathaniel, 1804-1864",Nathaniel Hawthorne,Nathaniel Hawthorne,0,0,0,...,historical,major_genre_non_domestic,nathaniel hawthorne,nathaniel hawthorne,0,NaN,0,0,non_domestic_genre,True
5,35,The Time Machine,Science fiction; Time travel -- Fiction; Dysto...,science fiction; time travel -- fiction; dysto...,"Wells, H. G. (Herbert George), 1866-1946",H. G. Wells,Herbert George Wells,0,0,1,...,science_fiction,major_genre_non_domestic,herbert george wells,h g wells,1,Edwardian,0,0,non_domestic_genre,True
6,36,The war of the worlds,Science fiction; War stories; Martians -- Fict...,science fiction; war stories; martians -- fict...,"Wells, H. G. (Herbert George), 1866-1946",H. G. Wells,Herbert George Wells,0,0,1,...,science_fiction,major_genre_non_domestic,herbert george wells,h g wells,1,Edwardian,0,0,non_domestic_genre,True
7,42,The strange case of Dr. Jekyll and Mr. Hyde,Science fiction; Horror tales; London (England...,science fiction; horror tales; london (england...,"Stevenson, Robert Louis, 1850-1894",Robert Louis Stevenson,Robert Louis Stevenson,0,0,1,...,science_fiction; gothic,major_genre_non_domestic,robert louis stevenson,robert louis stevenson,1,Victorian,0,0,non_domestic_genre,True
8,43,The strange case of Dr. Jekyll and Mr. Hyde,Science fiction; Horror tales; London (England...,science fiction; horror tales; london (england...,"Stevenson, Robert Louis, 1850-1894",Robert Louis Stevenson,Robert Louis Stevenson,0,0,1,...,science_fiction; gothic,major_genre_non_domestic,robert louis stevenson,robert louis stevenson,1,Victorian,0,0,non_domestic_genre,True
9,46,A Christmas Carol in Prose; Being a Ghost Stor...,Christmas stories; London (England) -- Fiction...,christmas stories; london (england) -- fiction...,"Dickens, Charles, 1812-1870; Leech, John, 1817...",Charles Dickens,Charles Dickens,1,0,0,...,gothic,domestic_and_genre,charles dickens,charles dickens,1,Victorian,0,0,domestic_adjacent_hybrid,True


### Author-name matching


In [107]:
def norm(s):
    """lowercase, strip accents, drop years and punctuation, collapse spaces"""
    if pd.isna(s):
        return ""
    s = unicodedata.normalize("NFKD", str(s))
    s = "".join(c for c in s if not unicodedata.combining(c))
    s = s.lower()
    s = re.sub(r"\[[^\]]*\]", " ", s)                  # [Illustrator]
    s = re.sub(r"\d{3,4}\s*[-?]\s*\d{0,4}", " ", s)    # 1866-1946
    s = re.sub(r"[^a-z\s]", " ", s)
    return re.sub(r"\s+", " ", s).strip()

# ── Wikidata side ───────────────────────────────────────────────
wd = per_writer.copy()
wd["name_norm"] = wd.writer.apply(norm)
wd = wd[wd.name_norm != ""]

# same normalized name → different QIDs. keep the most prominent, but count them
wd = wd.sort_values("linkcount", ascending=False)
collisions = wd[wd.name_norm.duplicated(keep=False)]
wd_lookup = wd.drop_duplicates("name_norm").set_index("name_norm")

# ── Gutenberg side: author-level table ──────────────────────────
authors = (gb.groupby("author_display", as_index=False)
             .agg(n_books=("gutenberg_id", "size"),
                  author_raw=("author_raw", "first"),
                  key_norm=("author_key_normalized", "first"),
                  disp_norm=("author_display_normalized", "first")))

authors["c_key"]  = authors.key_norm.apply(norm)
authors["c_disp"] = authors.disp_norm.apply(norm)

def match(row):
    for src, cand in (("display", row.c_disp), ("key", row.c_key)):
        if cand and cand in wd_lookup.index:
            return pd.Series([cand, src])
    return pd.Series([None, None])

authors[["hit_norm", "matched_via"]] = authors.apply(match, axis=1)

for col in ["writer", "birthYear", "deathYear", "gender",
            "tier", "all_tiers", "nationality", "is_uk", "linkcount"]:
    authors[f"wd_{col}"] = authors.hit_norm.map(wd_lookup[col])

# Gutenberg birth year, for verification only — not used to filter
authors["gb_birth"] = authors.author_raw.str.extract(r"(\d{4})").astype(float)

In [108]:
n_a, n_hit = len(authors), authors.hit_norm.notna().sum()
print(f"authors: {n_hit}/{n_a} matched ({n_hit/n_a:.1%})")
print(f"books:   {authors.loc[authors.hit_norm.notna(), 'n_books'].sum()}/{authors.n_books.sum()}")
print(f"\nmatched via:\n{authors.matched_via.value_counts(dropna=False)}")

print(f"\n── tier of matched authors ──")
print(authors.wd_tier.value_counts(dropna=False))
print(f"\nUK authors matched: {(authors.wd_is_uk == True).sum()}")

authors: 995/1906 matched (52.2%)
books:   4567/6618

matched via:
matched_via
None       911
display    897
key         98
Name: count, dtype: int64

── tier of matched authors ──
wd_tier
NaN         911
america     502
core        328
no_p27       64
europe       49
settler      39
ireland       7
other         4
colonial      2
Name: count, dtype: int64

UK authors matched: 337


### Match diagnostics


In [109]:
# ── 1. how big is the lookup table, and what does it cover? ──
print(f"per_writer rows: {len(per_writer)}, unique norm names: {len(wd_lookup)}")
print(per_writer.birthYear.describe())
print(per_writer.tier.value_counts())

per_writer rows: 61445, unique norm names: 61073
count    61445.000000
mean      1867.214647
std         57.746592
min       1650.000000
25%       1839.000000
50%       1885.000000
75%       1911.000000
max       1930.000000
Name: birthYear, dtype: float64
tier
europe      22474
america     12397
core         7947
no_p27       7480
other        6535
settler      2300
colonial     1909
ireland       403
Name: count, dtype: int64


In [110]:
# ── 2. known-answer probe: absent from per_writer, or present under another form? ──
wd_surnames = {}
for n in wd_lookup.index:
    t = n.split()
    if len(t) >= 2:
        wd_surnames.setdefault(t[-1], []).append(n)

def probe(name):
    n = norm(name)
    sur = n.split()[-1] if len(n.split()) >= 2 else n
    return {"query": n,
            "exact": n in wd_lookup.index,
            "same_surname": wd_surnames.get(sur, [])[:6]}

for name in ["Jules Verne", "Charles Dickens", "Rudyard Kipling",
             "George Manville Fenn", "Percy F. Westerman", "Harold Bindloss",
             "Stanley John Weyman", "Gordon Stables", "Louis Tracy",
             "Mark Twain", "Frederik Pohl", "Leigh Brackett"]:
    r = probe(name)
    print(f"{name:28} exact={r['exact']!s:5} surname_matches={r['same_surname']}")

Jules Verne                  exact=True  surname_matches=['jules verne', 'michel verne']
Charles Dickens              exact=True  surname_matches=['charles dickens', 'catherine dickens', 'monica dickens', 'mary angela dickens', 'mary dickens', 'augustus dickens']
Rudyard Kipling              exact=True  surname_matches=['rudyard kipling', 'alice macdonald kipling']
George Manville Fenn         exact=True  surname_matches=['george manville fenn', 'ellenor fenn', 'john fenn', 'joseph finch fenn', 'forrest fenn']
Percy F. Westerman           exact=True  surname_matches=['gerhart von westerman', 'percy f westerman', 'john f c westerman']
Harold Bindloss              exact=False surname_matches=['harold edward bindloss']
Stanley John Weyman          exact=False surname_matches=['stanley j weyman']
Gordon Stables               exact=False surname_matches=['william gordon stables']
Louis Tracy                  exact=True  surname_matches=['antoine destutt de tracy', 'mona innis tracy', 'louis

In [111]:
# ── 3. split the 1255 into "absent from Wikidata" vs "name-form mismatch" ──
SUFFIX = re.compile(r"\b(?:jr|sr|ii|iii|iv|mrs|mr|ms|dr|rev|sir|lady|lord)\b")

def surname_of(s):
    t = SUFFIX.sub(" ", s).split()
    return t[-1] if len(t) >= 2 else (t[0] if t else "")

un = authors[authors.hit_norm.isna()].copy()
un["surname"] = un.c_disp.apply(surname_of)
un["surname_in_wd"] = un.surname.isin(wd_surnames)

print(un.surname_in_wd.value_counts())
print(f"\nbooks affected: name-form candidates "
      f"{un.loc[un.surname_in_wd, 'n_books'].sum()}, "
      f"likely absent {un.loc[~un.surname_in_wd, 'n_books'].sum()}")

print("\n── surname exists in Wikidata → probably a name-form miss ──")
print(un[un.surname_in_wd].nlargest(25, "n_books")
      [["author_display", "author_raw", "n_books"]].to_string(index=False))

print("\n── surname absent → probably not in per_writer at all ──")
print(un[~un.surname_in_wd].nlargest(25, "n_books")
      [["author_display", "author_raw", "n_books"]].to_string(index=False))

surname_in_wd
True     712
False    199
Name: count, dtype: int64

books affected: name-form candidates 1679, likely absent 372

── surname exists in Wikidata → probably a name-form miss ──
             author_display                                                                                     author_raw  n_books
            Victor Appleton                                                                               Appleton, Victor       33
        Stanley R. Matthews                                                                           Matthews, Stanley R.       32
            Mildred A. Wirt                                                Wirt, Mildred A. (Mildred Augustine), 1905-2002       26
            Nicholas Carter                                                                  Carter, Nicholas (House name)       25
             Gordon Stables                                                                     Stables, Gordon, 1840-1910       21
          Horatio 

In [114]:
# ── 4. is the 1650–1930 birth filter cutting real authors? ──
un["gb_birth"] = un.author_raw.str.extract(r"(\d{4})").astype(float)

print(f"no birth year in author_raw: {un.gb_birth.isna().sum()}")
print(f"born after 1930:  {(un.gb_birth > 1930).sum()}")
print(f"born before 1650: {(un.gb_birth < 1650).sum()}")
print(f"inside range:     {un.gb_birth.between(1700, 1910).sum()}")

print("\n── unmatched, born after 1930 ──")
print(un[un.gb_birth > 1930].nlargest(20, "n_books")
      [["author_display", "gb_birth", "n_books"]].to_string(index=False))

no birth year in author_raw: 185
born after 1930:  32
born before 1650: 5
inside range:     606

── unmatched, born after 1930 ──
     author_display  gb_birth  n_books
      Albert dePina    1957.0        6
  Robert Silverberg    1935.0        6
       Algis Budrys    1931.0        4
       Jack Sharkey    1931.0        4
    Josephine Chase    1931.0        4
      Thomas Hoover    1941.0        4
      Cory Doctorow    1971.0        3
 Donald E. Westlake    1933.0        3
Laurence M. Janifer    1933.0        3
           Ben Bova    1932.0        2
   Robert J. Sawyer    1960.0        2
        Robert Shea    1933.0        2
   Samuel R. Delany    1942.0        2
     Alexei Panshin    1940.0        1
   Andrew J. Offutt    1934.0        1
         Ann Bannon    1932.0        1
    David E. Fisher    1932.0        1
          Don Berry    1932.0        1
       Don Thompson    1935.0        1
     Gerald W. Page    1939.0        1


In [115]:
STOP = re.compile(r"^(?:jr|sr|ii|iii|iv|mr|mrs|ms|dr|rev|sir|lady|lord"
                  r"|baron|baroness|dame|st|the)$")

def toks(s):
    """normalized tokens with honorifics/suffixes removed"""
    return [t for t in norm(s).split() if not STOP.match(t)]

def k_exact(s):
    t = toks(s)
    return " ".join(t) if len(t) >= 2 else ""

def k_firstlast(s):
    t = toks(s)
    if len(t) < 2 or len(t[0]) == 1:      # initials go to the next layer
        return ""
    return f"{t[0]}|{t[-1]}"

def k_initlast(s):
    t = toks(s)
    return f"{t[0][0]}|{t[-1]}" if len(t) >= 2 else ""

LAYERS = [("exact", k_exact), ("firstlast", k_firstlast), ("initlast", k_initlast)]

In [116]:
wd = per_writer.copy()
wd["_n"] = wd.writer.apply(norm)
wd = wd[wd._n != ""]

lookups = {}
for name, f in LAYERS:
    wd[f"k_{name}"] = wd.writer.apply(f)
    sub = wd[wd[f"k_{name}"] != ""]
    lookups[name] = (sub.groupby(f"k_{name}")
                        .apply(lambda g: list(zip(g.s, g.writer, g.birthYear)))
                        .to_dict())

/tmp/ipykernel_1935/3955457286.py:10: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: list(zip(g.s, g.writer, g.birthYear)))
/tmp/ipykernel_1935/3955457286.py:10: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: list(zip(g.s, g.writer, g.birthYear)))
/tmp/ipykernel_1935/3955457286.py:10: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior i

In [117]:
TOL = 2

def resolve(cands, gb_birth):
    """cands: [(qid, writer, birthYear)] -> (qid, writer, status) or None"""
    if gb_birth and not pd.isna(gb_birth):
        ok = [c for c in cands if pd.notna(c[2]) and abs(c[2] - gb_birth) <= TOL]
        if len(ok) == 1:
            return ok[0][0], ok[0][1], "year_ok"
        if len(ok) > 1:
            return ok[0][0], ok[0][1], "year_ok_ambiguous"
        return None                       # year present and disagrees -> reject
    if len(cands) == 1:                   # no year to check; accept only if unique
        return cands[0][0], cands[0][1], "no_year_unique"
    return None

def match_row(row):
    for lname, f in LAYERS:
        for cand in (row.c_disp, row.c_key):
            if not cand:
                continue
            k = f(cand)
            if not k:
                continue
            hits = lookups[lname].get(k)
            if not hits:
                continue
            r = resolve(hits, row.gb_birth)
            if r:
                return pd.Series([r[0], r[1], lname, r[2], len(hits)])
    return pd.Series([None, None, None, None, 0])

authors["gb_birth"] = authors.author_raw.str.extract(r"(\d{4})").astype(float)
authors[["wd_qid", "wd_writer", "match_layer", "match_status", "n_cand"]] = \
    authors.apply(match_row, axis=1)

info = wd.drop_duplicates("s").set_index("s")
for col in ["birthYear", "deathYear", "gender", "tier",
            "all_tiers", "nationality", "is_uk"]:
    authors[f"wd_{col}"] = authors.wd_qid.map(info[col])

In [118]:
real = authors[authors.author_raw.str.contains(r"\d{4}", na=False)]   # drop house names
print(f"all authors:  {authors.wd_qid.notna().sum()}/{len(authors)}")
print(f"with a year:  {real.wd_qid.notna().sum()}/{len(real)}")
print(f"books:        {authors.loc[authors.wd_qid.notna(), 'n_books'].sum()}/{authors.n_books.sum()}")

print("\n", pd.crosstab(authors.match_layer, authors.match_status, dropna=False))
print("\n", authors.wd_tier.value_counts(dropna=False))
print(f"\nUK: {(authors.wd_is_uk == True).sum()}")

all authors:  1140/1906
with a year:  1095/1716
books:        4987/6618

 match_status  no_year_unique  year_ok  year_ok_ambiguous  NaN
match_layer                                                  
exact                      5      958                  0    0
firstlast                  4      103                  0    0
initlast                  36       33                  1    0
NaN                        0        0                  0  766

 wd_tier
NaN         766
america     570
core        377
no_p27       79
europe       55
settler      42
ireland      10
other         5
colonial      2
Name: count, dtype: int64

UK: 389


In [119]:
print("── new matches from firstlast / initlast ──")
print(authors[authors.match_layer.isin(["firstlast", "initlast"])]
      .nlargest(40, "n_books")
      [["author_display", "wd_writer", "gb_birth", "wd_birthYear",
        "match_layer", "match_status", "n_books"]].to_string(index=False))

print("\n── accepted without year verification (riskiest) ──")
print(authors[authors.match_status == "no_year_unique"]
      .nlargest(20, "n_books")
      [["author_display", "wd_writer", "match_layer", "n_books"]].to_string(index=False))

── new matches from firstlast / initlast ──
                       author_display                                    wd_writer  gb_birth  wd_birthYear match_layer   match_status  n_books
                        Emmuska Orczy                                   Emma Orczy    1865.0        1865.0    initlast        year_ok       18
           Elizabeth Cleghorn Gaskell                            Elizabeth Gaskell    1810.0        1810.0   firstlast        year_ok       17
                  Stanley John Weyman                            Stanley J. Weyman    1855.0        1855.0   firstlast        year_ok       15
                       Laura Lee Hope                                  Ludvig Hope       NaN        1871.0    initlast no_year_unique       14
          Mary Wollstonecraft Shelley                                 Mary Shelley    1797.0        1797.0   firstlast        year_ok       14
                       Clarence Young                                Charles Young       NaN      